#loading the data set, printing the first few rows to make sure it worked and see the data

In [1]:
import pandas as pd

url = "https://drive.google.com/uc?export=download&id=1O7Cqs2rK27s1Gqi3pmkJpYtXnc4tQphc"
df = pd.read_csv(url, encoding="latin1", low_memory=False)

df.head()

,Sentence #,Word,POS,Tag
0,Sentence: 1,Thousands,NNS,O
1,NaN,of,IN,O
2,NaN,demonstrators,NNS,O
3,NaN,have,VBP,O
4,NaN,marched,VBN,O


In [2]:
#checking the column names so the code is accurate
df.columns

Index(['Sentence #', 'Word', 'POS', 'Tag'], dtype='object')

#all the pieces of the data frame are saved into sentences. each word in the sentence, with its pos and tag is grouped and saved until a new sentence is started.

In [11]:
sentences = []
current_sentence = []

for _, row in df.iterrows():
    if isinstance(row['Sentence #'], str) and row['Sentence #'].startswith("Sentence:"):
        if current_sentence:
            sentences.append(current_sentence)
            current_sentence = []
    current_sentence.append((row['Word'], row['POS'], row['Tag']))

# Add last sentence
if current_sentence:
    sentences.append(current_sentence)


In [12]:
df.head()

,Sentence #,Word,POS,Tag
0,Sentence: 1,Thousands,NNS,O
1,NaN,of,IN,O
2,NaN,demonstrators,NNS,O
3,NaN,have,VBP,O
4,NaN,marched,VBN,O


###looking for all unique/possible tags

In [13]:
df["Tag"].unique()

array(['O', 'B-geo', 'B-gpe', 'B-per', 'I-geo', 'B-org', 'I-org', 'B-tim',
       'B-art', 'I-art', 'I-per', 'I-gpe', 'I-tim', 'B-nat', 'B-eve',
       'I-eve', 'I-nat'], dtype=object)

The following cell takes in the sentences and creates a dictionary about the information regarding each word and the previous and next word. it also includes information about special markers for beginning and end of sentences. This information is used to train the CRF model.

In [16]:
def word2features(sent, i):
    word = sent[i][0]
    postag = sent[i][1]
    word = str(word)

    features = {
        "bias": 1.0,
        "word.lower()": word.lower(),
        "word.isupper()": word.isupper(),
        "word.istitle()": word.istitle(),
        "word.isdigit()": word.isdigit(),
        "postag": postag,
        "postag[:2]": postag[:2],
    }

    if i > 0:
        word1 = str(sent[i-1][0])
        postag1 = sent[i-1][1]
        features.update({
            "-1:word.lower()": word1.lower(),
            "-1:word.istitle()": word1.istitle(),
            "-1:word.isupper()": word1.isupper(),
            "-1:postag": postag1,
            "-1:postag[:2]": postag1[:2],
        })
    else:
        features["BOS"] = True

    if i < len(sent)-1:
        word1 = str(sent[i+1][0])
        postag1 = sent[i+1][1]
        features.update({
            "+1:word.lower()": word1.lower(),
            "+1:word.istitle()": word1.istitle(),
            "+1:word.isupper()": word1.isupper(),
            "+1:postag": postag1,
            "+1:postag[:2]": postag1[:2],
        })
    else:
        features["EOS"] = True

    return features


def sent2features(sent):
    return [word2features(sent, i) for i in range(len(sent))]


def sent2labels(sent):
    return [label for (_, _, label) in sent]


   


splitting the data into 80%-20% train and test data.

In [19]:
from sklearn.model_selection import train_test_split

X = [sent2features(s) for s in sentences]
y = [sent2labels(s) for s in sentences]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)


Training and creating the crf model. I set the words training is complete to notify when the training is complete.

In [20]:
!pip install sklearn-crfsuite
import sklearn_crfsuite

crf = sklearn_crfsuite.CRF(
    algorithm="lbfgs",
    c1=0.1,
    c2=0.1,
    max_iterations=100,
    all_possible_transitions=True,
)

print("Training CRF model... this may take a few minutes.")
crf.fit(X_train, y_train)
print("Training complete.")



Defaulting to user installation because normal site-packages is not writeable
Looking in links: /usr/share/pip-wheels
Training CRF model... this may take a few minutes.
Training complete.


Model evaluation

In [21]:
from sklearn_crfsuite import metrics

y_pred = crf.predict(X_test)

print("Model Evaluation:")
print(metrics.flat_classification_report(y_test, y_pred))

Model Evaluation:
              precision    recall  f1-score   support

       B-art       0.35      0.14      0.20        86
       B-eve       0.46      0.30      0.36        60
       B-geo       0.86      0.91      0.88      7664
       B-gpe       0.97      0.94      0.95      3175
       B-nat       0.62      0.32      0.42        50
       B-org       0.80      0.74      0.77      3913
       B-per       0.84      0.83      0.84      3389
       B-tim       0.93      0.88      0.90      4049
       I-art       0.04      0.02      0.02        58
       I-eve       0.42      0.25      0.31        53
       I-geo       0.82      0.79      0.80      1450
       I-gpe       0.80      0.62      0.70        39
       I-nat       0.44      0.33      0.38        12
       I-org       0.83      0.80      0.82      3315
       I-per       0.85      0.91      0.88      3445
       I-tim       0.83      0.77      0.79      1300
           O       0.99      0.99      0.99    176877

    accu

Installing spacy

In [22]:
!pip install spacy
!python -m spacy download en_core_web_sm

Defaulting to user installation because normal site-packages is not writeable
Looking in links: /usr/share/pip-wheels
Defaulting to user installation because normal site-packages is not writeable
Looking in links: /usr/share/pip-wheels
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 70.9 MB/s  0:00:0073.4 MB/s eta 0:00:01
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')


Importing the spacy model

In [23]:
import spacy
nlp = spacy.load("en_core_web_sm")

Running the sentence through the spacy model. This model detects and prints all the named entities.

In [24]:
sentence = "Fourteen days ago, Emperor Palpatine left San Diego, CA for Tatooine to follow Luke Skywalker."
doc = nlp(sentence)

for ent in doc.ents:
    print(ent.text, ent.label_)

Fourteen days ago DATE
Palpatine PERSON
San Diego GPE
CA WORK_OF_ART
Tatooine PERSON
Luke Skywalker PERSON


We are running the same sentence through the Ner, CRf pipeline that was created a few cells back

In [25]:
import nltk
from nltk import word_tokenize, pos_tag

sentence = "Fourteen days ago, Emperor Palpatine left San Diego, CA for Tatooine to follow Luke Skywalker."

# Tokenize and POS-tag
tokens = word_tokenize(sentence)
pos_tags = pos_tag(tokens)

# Convert to CRF feature format
test_sent = [(word, pos, 'O') for word, pos in pos_tags]  # dummy tags
X_test_sent = [sent2features(test_sent)]

# Predict
y_pred_sent = crf.predict(X_test_sent)[0]

# Display results
for word, tag in zip(tokens, y_pred_sent):
    print(f"{word:15} {tag}")

Fourteen        B-org
days            O
ago             O
,               O
Emperor         B-per
Palpatine       I-per
left            O
San             B-geo
Diego           I-geo
,               O
CA              B-org
for             I-org
Tatooine        I-org
to              O
follow          O
Luke            B-org
Skywalker       I-org
.               O


Evaluating this model

In [26]:
from sklearn_crfsuite import metrics

labels = list(crf.classes_)
labels.remove('O')

y_pred = crf.predict(X_test)

print(metrics.flat_classification_report(y_test, y_pred, labels=labels))

              precision    recall  f1-score   support

       B-gpe       0.97      0.94      0.95      3175
       B-geo       0.86      0.91      0.88      7664
       B-org       0.80      0.74      0.77      3913
       I-org       0.83      0.80      0.82      3315
       I-geo       0.82      0.79      0.80      1450
       B-tim       0.93      0.88      0.90      4049
       B-per       0.84      0.83      0.84      3389
       I-per       0.85      0.91      0.88      3445
       B-eve       0.46      0.30      0.36        60
       I-eve       0.42      0.25      0.31        53
       I-tim       0.83      0.77      0.79      1300
       B-nat       0.62      0.32      0.42        50
       I-nat       0.44      0.33      0.38        12
       I-gpe       0.80      0.62      0.70        39
       B-art       0.35      0.14      0.20        86
       I-art       0.04      0.02      0.02        58

   micro avg       0.86      0.85      0.86     32058
   macro avg       0.68   

Compare the spacy model to the homemade CRF model


At first the scores on the top of the spacy model were a lot lower than the scores of the crf, but that isn't necessarily the best place to get the overall accuracy and precision score. The f1-score is the most important because it is the accuracy and precision combined. The micro f1 score is the average f1 score but all entities have the same weight so i doesn't show the most precise information. The macro fi-score is fair across all entity types. either way the spacy model scored higher, at .97 and .65 as apposed to .86 and .63. The macro avg of the f1 scores are only 2% off, which isn't such a big difference.

